In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("auto-mpg.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.replace('?', np.nan, inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df['horsepower'] = pd.to_numeric(df['horsepower'])
df['horsepower'].fillna(df['horsepower'].median(), inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(df['mpg'], kde=True)

plt.title("MPG Distribution")
plt.show()

The MPG distribution is slightly right-skewed.
Most vehicles have MPG values between 15 and 30.
A few vehicles have very high MPG values, indicating possible outliers or highly fuel-efficient cars.
The target variable appears reasonably distributed for regression modeling.

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
for column in df.select_dtypes(include=np.number):

    plt.figure(figsize=(6,3))

    sns.boxplot(x=df[column])

    plt.title(column)

    plt.show()

In [ ]:
df['brand'] = df['car name'].apply(lambda x: x.split()[0])

In [ ]:
df['brand'].value_counts()

In [ ]:
df = pd.get_dummies(
    df,
    columns=['brand'],
    drop_first=True
)

In [ ]:
df.drop('car name', axis=1, inplace=True)

In [ ]:
df = pd.get_dummies(
    df,
    columns=['origin'],
    drop_first=True
)

In [ ]:
df.head()

In [ ]:
df.rename(columns={'model year': 'model_year'}, inplace=True)

In [ ]:
X = df.drop('mpg', axis=1)

y = df['mpg']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Impute numeric missing values before scaling/modeling
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

In [ ]:
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_mae = mean_absolute_error(y_test, lr_pred)

lr_mse = mean_squared_error(y_test, lr_pred)

lr_rmse = np.sqrt(lr_mse)

lr_r2 = r2_score(y_test, lr_pred)

print("MAE :", lr_mae)
print("MSE :", lr_mse)
print("RMSE :", lr_rmse)
print("R2 Score :", lr_r2)

In [ ]:
#ridge

from sklearn.linear_model import Ridge
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
ridge_pred = ridge_model.predict(X_test_scaled)
ridge_mae = mean_absolute_error(y_test, ridge_pred)

ridge_mse = mean_squared_error(y_test, ridge_pred)

ridge_rmse = np.sqrt(ridge_mse)

ridge_r2 = r2_score(y_test, ridge_pred)

print("MAE :", ridge_mae)
print("MSE :", ridge_mse)
print("RMSE :", ridge_rmse)
print("R2 Score :", ridge_r2)

In [ ]:
#svm
from sklearn.svm import SVR
svr_model = SVR(
    kernel='rbf'
)
svr_model.fit(X_train_scaled, y_train)
svr_pred = svr_model.predict(X_test_scaled)
svr_mae = mean_absolute_error(y_test, svr_pred)

svr_mse = mean_squared_error(y_test, svr_pred)

svr_rmse = np.sqrt(svr_mse)

svr_r2 = r2_score(y_test, svr_pred)

print("MAE :", svr_mae)
print("MSE :", svr_mse)
print("RMSE :", svr_rmse)
print("R2 Score :", svr_r2)

In [ ]:
#random forest regression

from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_mae = mean_absolute_error(y_test, rf_pred)

rf_mse = mean_squared_error(y_test, rf_pred)

rf_rmse = np.sqrt(rf_mse)

rf_r2 = r2_score(y_test, rf_pred)

print("MAE :", rf_mae)
print("MSE :", rf_mse)
print("RMSE :", rf_rmse)
print("R2 Score :", rf_r2)

In [ ]:
results = pd.DataFrame({

    'Model': [
        'Linear Regression',
        'Ridge Regression',
        'SVR',
        'Random Forest'
    ],

    'R2 Score': [
        lr_r2,
        ridge_r2,
        svr_r2,
        rf_r2
    ],

    'RMSE': [
        lr_rmse,
        ridge_rmse,
        svr_rmse,
        rf_rmse
    ]
})

results

In [ ]:
results.sort_values(
    by='R2 Score',
    ascending=False
)

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(
    x='Model',
    y='R2 Score',
    data=results
)

plt.title("Model Comparison")

plt.show()

In [ ]:
#hyperperameter tuning
from sklearn.model_selection import GridSearchCV
param_grid = {

    'n_estimators': [50, 100, 200],

    'max_depth': [5, 10, 15],

    'min_samples_split': [2, 5],

    'min_samples_leaf': [1, 2]
}
rf = RandomForestRegressor(
    random_state=42
)
grid_search = GridSearchCV(

    estimator=rf,

    param_grid=param_grid,

    cv=5,

    scoring='r2',

    n_jobs=-1,

    verbose=2
)
grid_search.fit(X_train_scaled, y_train)
grid_search.best_params_
grid_search.best_score_
best_rf = grid_search.best_estimator_
best_pred = best_rf.predict(X_test_scaled)
final_r2 = r2_score(y_test, best_pred)

final_rmse = np.sqrt(
    mean_squared_error(y_test, best_pred)
)

print("Final R2 Score:", final_r2)

print("Final RMSE:", final_rmse)

In [ ]:
grid_search.best_params_
grid_search.best_score_
best_rf = grid_search.best_estimator_
best_pred = best_rf.predict(X_test_scaled)
final_r2 = r2_score(y_test, best_pred)

print(final_r2)

In [ ]:
grid_search.best_params_

In [ ]:
##Feature Importance

importance = best_rf.feature_importances_
feature_importance = pd.DataFrame({

    'Feature': X.columns,

    'Importance': importance

})
feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

feature_importance


plt.figure(figsize=(12,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance
)

plt.title("Feature Importance")

plt.show()

Weight and horsepower were the most influential features in predicting MPG.
Heavier cars generally showed lower fuel efficiency.

In [ ]:
## pipelinea

from sklearn.pipeline import Pipeline
final_pipeline = Pipeline([

    ('scaler', StandardScaler()),

    ('model', RandomForestRegressor(

        max_depth=15,

        min_samples_leaf=1,

        min_samples_split=2,

        n_estimators=50,

        random_state=42
    ))
])

final_pipeline.fit(X_train, y_train)
pipeline_pred = final_pipeline.predict(X_test)
pipeline_r2 = r2_score(y_test, pipeline_pred)

print("Pipeline R2 Score:", pipeline_r2)

In [ ]:
import joblib

joblib.dump(final_pipeline, 'final_pipeline.pkl')